# SahajMobile · Cohort-Based MOIC Curve Analysis

Build a cohort-based MOIC (multiple on invested capital) curve from raw installment payment data.

**Formula reference**

| Metric | Formula |
|---|---|
| Payment MOIC | Period collections ÷ Cohort total advance |
| Cumulative MOIC | Running collections ÷ Cohort total advance |

**Workflow** — the blocks below clean the data, calculate the cohort summary tables, and write the deliverables to disk.

## Setup — imports & configuration

All pipeline logic lives in `sahajmobile_moic.py`; this notebook imports it so the
two entry points cannot drift apart. Run the notebook from the project folder so
the module and the CSV are importable/resolvable.

In [1]:
%matplotlib inline
from IPython.display import display

from sahajmobile_moic import (
    build_executive_insights,
    build_tables,
    compute_moic,
    load_and_clean,
    plot_executive_dashboard,
    plot_moic_curves,
    plot_net_moic_curves,
    resolve_paths,
)

# Paths come from SAHAJMOBILE_INPUT / SAHAJMOBILE_OUTPUT, else the defaults next
# to sahajmobile_moic.py. The empty list keeps kernel argv out of path resolution.
in_path, out_path = resolve_paths([])
print(f"  Input CSV : {in_path}")
print(f"  Output dir: {out_path}")

  Input CSV : /home/ubuntu/repos/Cohort-based-MOIC-curve-analysis-for-SahajMobile-/Installment_shorter_sampled.csv
  Output dir: /tmp/nbout


## Section 1 · Load & Clean

`load_and_clean` standardises column names and types, removes exact duplicates,
and separates anomalous rows.

In [2]:
cleaned, excluded = load_and_clean(in_path)

print("\n── Cleaned dataset (first 10 rows) ──────────────────────────")
display(cleaned.head(10))

if len(excluded):
    print("\n── Excluded rows (with reason) ────────────────────────────")
    display(excluded[["Asset_ID", "Origination_Date", "Payment_Date", "Payment_Amount", "Exclusion_Reason"]])

  Raw rows            :  11,894
  Exact duplicates    :     638  (removed)
  Anomalous/excluded  :       4  rows
  Cleaned rows        :  11,252
  Unique cohorts      :      15
  Unique assets       :     351

  Exclusion breakdown:
       2  Payment_Date before Origination_Date
       1  Sentinel / Epoch Payment_Date (1970-01-01)
       1  Months on Book > 24 — likely data entry error

── Cleaned dataset (first 10 rows) ──────────────────────────


,Asset_ID,Origination_Date,Total_Advance,Total_EMI,Payment_Date,Payment_Amount,Months_on_Book,Cohort
0,264,2024-01-31,17160.0,20540.0,2024-02-07,860.0,1,2024-01
1,264,2024-01-31,17160.0,20540.0,2024-02-14,860.0,1,2024-01
2,264,2024-01-31,17160.0,20540.0,2024-02-21,860.0,1,2024-01
3,264,2024-01-31,17160.0,20540.0,2024-02-28,860.0,1,2024-01
4,264,2024-01-31,17160.0,20540.0,2024-03-06,860.0,2,2024-01
5,264,2024-01-31,17160.0,20540.0,2024-03-13,860.0,2,2024-01
6,264,2024-01-31,17160.0,20540.0,2024-03-20,860.0,2,2024-01
7,264,2024-01-31,17160.0,20540.0,2024-03-27,860.0,2,2024-01
8,264,2024-01-31,17160.0,20540.0,2024-04-03,860.0,3,2024-01
9,264,2024-01-31,17160.0,20540.0,2024-04-10,9000.0,3,2024-01



── Excluded rows (with reason) ────────────────────────────


,Asset_ID,Origination_Date,Payment_Date,Payment_Amount,Exclusion_Reason
11892,24105,2026-01-06,1970-01-01,2500.0,Sentinel / Epoch Payment_Date (1970-01-01)
10085,1454,2024-11-18,1970-01-03,1.0,Payment_Date before Origination_Date
10207,1465,2024-11-20,2024-07-02,0.0,Payment_Date before Origination_Date
11888,13747,2025-10-24,2027-11-27,2000.0,Months on Book > 24 — likely data entry error


## Section 2 · MOIC Computation

`compute_moic` aggregates collections by cohort × months-on-book and computes
Payment MOIC, Cumulative MOIC and Net MOIC.

In [3]:
moic_df, cohort_meta = compute_moic(cleaned)

print("\n── Cohort metadata ──────────────────────────────────────────")
display(cohort_meta)

print("\n── MOIC long-form table (first 15 rows) ────────────────────")
display(moic_df.head(15))

  Cohorts in MOIC table : 15
  Max Months on Book    : 21
  Max Cumulative MOIC   : 1.3903×  (cohort 2024-09)
  Max Net MOIC          : +0.3903×  (cohort 2024-09)

── Cohort metadata ──────────────────────────────────────────


,Cohort,Asset_Count,Cohort_Total_Advance
0,2024-01,6,102945.0
1,2024-02,11,136095.0
2,2024-03,64,795665.0
3,2024-04,32,368935.0
4,2024-05,16,224805.0
5,2024-06,32,447855.0
6,2024-07,20,286900.0
7,2024-08,28,325990.0
8,2024-09,37,408306.0
9,2024-10,37,461204.0



── MOIC long-form table (first 15 rows) ────────────────────


,Cohort,Months_on_Book,Period_Collections,Cohort_Total_Advance,Payment_MOIC,Cumulative_Collections,Cumulative_MOIC,Net_MOIC
0,2024-01,1,23498.0,102945.0,0.228258,23498.0,0.228258,-0.771742
1,2024-01,2,24097.0,102945.0,0.234076,47595.0,0.462334,-0.537666
2,2024-01,3,39580.0,102945.0,0.384477,87175.0,0.846811,-0.153189
3,2024-01,4,12689.0,102945.0,0.123260,99864.0,0.970071,-0.029929
4,2024-01,5,11843.0,102945.0,0.115042,111707.0,1.085113,0.085113
5,2024-01,6,4950.0,102945.0,0.048084,116657.0,1.133197,0.133197
6,2024-01,7,0.0,102945.0,0.000000,116657.0,1.133197,0.133197
7,2024-01,8,0.0,102945.0,0.000000,116657.0,1.133197,0.133197
8,2024-01,9,2493.0,102945.0,0.024217,119150.0,1.157414,0.157414
9,2024-02,0,7520.0,136095.0,0.055256,7520.0,0.055256,-0.944744


## Section 3 · Summary Tables

`build_tables` returns the cohort KPI summary plus wide pivots of Cumulative and
Payment MOIC; `build_executive_insights` adds the CTO-facing view.

In [4]:
summary, cum_matrix, pay_matrix = build_tables(moic_df, cohort_meta)
print("── Cohort summary table ────────────────────────────────────")
display(summary)

print("\n── Cumulative MOIC matrix (cohort × MOB) ───────────────────")
display(cum_matrix)

print("\n── Payment MOIC matrix (cohort × MOB) ─────────────────────")
display(pay_matrix)

print("\n[3b] Building Executive Insights ...")
executive_insights, portfolio_summary = build_executive_insights(moic_df, summary)
print("── Executive Insights ─────────────────────────────────────")
display(executive_insights)

print("\n── Portfolio Summary ───────────────────────────────────────")
display(portfolio_summary)

outputs = {
    "cleaned_installments.csv"   : cleaned,
    "excluded_rows.csv"          : excluded,
    "moic_table.csv"             : moic_df,
    "cohort_summary.csv"         : summary,
    "executive_insights.csv"     : executive_insights,
    "portfolio_summary.csv"      : portfolio_summary,
    "cumulative_moic_matrix.csv" : cum_matrix,
    "payment_moic_matrix.csv"    : pay_matrix,
}
for filename, df in outputs.items():
    df.to_csv(f"{out_path}/{filename}", index=False)
    print(f"  ✓  {filename:<38}  ({len(df):,} rows)")

── Cohort summary table ────────────────────────────────────


,Cohort,Asset_Count,Cohort_Total_Advance,Total_Collected,Max_Months_on_Book,Max_Cumulative_MOIC,Max_Net_MOIC,Avg_Payment_MOIC
0,2024-01,6,102945.0,119150.0,9,1.157414,0.157414,0.128602
1,2024-02,11,136095.0,181074.0,8,1.330497,0.330497,0.147833
2,2024-03,64,795665.0,1030222.0,12,1.294794,0.294794,0.099600
3,2024-04,32,368935.0,506270.0,11,1.372247,0.372247,0.114354
4,2024-05,16,224805.0,293500.0,9,1.305576,0.305576,0.130558
5,2024-06,32,447855.0,583578.0,21,1.303051,0.303051,0.059230
6,2024-07,20,286900.0,381188.0,9,1.328644,0.328644,0.132864
7,2024-08,28,325990.0,445617.0,16,1.366965,0.366965,0.080410
8,2024-09,37,408306.0,567657.0,17,1.390273,0.390273,0.077237
9,2024-10,37,461204.0,614090.0,15,1.331493,0.331493,0.083218



── Cumulative MOIC matrix (cohort × MOB) ───────────────────


,Cohort,MOB_0,MOB_1,MOB_2,MOB_3,MOB_4,MOB_5,MOB_6,MOB_7,MOB_8,...,MOB_12,MOB_13,MOB_14,MOB_15,MOB_16,MOB_17,MOB_18,MOB_19,MOB_20,MOB_21
0,2024-01,NaN,0.228258,0.462334,0.846811,0.970071,1.085113,1.133197,1.133197,1.133197,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-02,0.055256,0.284845,0.543363,0.749572,0.991690,1.201800,1.304537,1.304537,1.330497,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-03,0.097275,0.327693,0.556440,0.778335,0.977024,1.166996,1.210851,1.261189,1.276129,...,1.294794,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04,0.100129,0.357055,0.575980,0.799959,1.029883,1.206936,1.276008,1.312413,1.328879,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-05,0.082649,0.317827,0.542741,0.741709,0.949774,1.194102,1.287605,1.296524,1.300305,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-06,0.135437,0.354228,0.589803,0.784051,0.972136,1.110911,1.201864,1.240196,1.250579,...,1.268174,1.270407,1.275989,1.277552,1.277552,1.282018,1.286483,1.292974,1.299032,1.303051
6,2024-07,0.140415,0.376978,0.607801,0.839826,1.031854,1.256239,1.269306,1.314793,1.314793,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-08,0.045379,0.288509,0.540458,0.760560,0.979183,1.196816,1.300798,1.317362,1.323406,...,1.328774,1.336289,1.354695,1.357763,1.366965,NaN,NaN,NaN,NaN,NaN
8,2024-09,0.078250,0.317265,0.544187,0.789508,0.988705,1.165454,1.297880,1.316248,1.339515,...,1.368231,1.378028,1.387824,1.387824,1.387824,1.390273,NaN,NaN,NaN,NaN
9,2024-10,0.088967,0.336602,0.598824,0.817630,0.990590,1.150996,1.222990,1.253296,1.255464,...,1.296715,1.307122,1.323774,1.331493,NaN,NaN,NaN,NaN,NaN,NaN



── Payment MOIC matrix (cohort × MOB) ─────────────────────


...


[3b] Building Executive Insights ...
── Executive Insights ─────────────────────────────────────


,Cohort,Recovery_Status,Seasoned_12M,Latest_MOB,Latest_Cumulative_MOIC,Months_to_Breakeven,Months_to_Target_1_3x,Collection_Share_Pct,Advance_Share_Pct,Max_Cumulative_MOIC
8,2024-09,Beyond target,True,17,1.390273,5,7,10.022193,9.083004,1.390273
3,2024-04,Beyond target,False,11,1.372247,4,7,8.938383,8.207173,1.372247
7,2024-08,Beyond target,True,16,1.366965,5,6,7.867532,7.251837,1.366965
9,2024-10,Beyond target,True,15,1.331493,5,13,10.841984,10.259751,1.331493
1,2024-02,Beyond target,False,8,1.330497,5,6,3.196928,3.027512,1.330497
6,2024-07,Beyond target,False,9,1.328644,4,7,6.730014,6.382257,1.328644
4,2024-05,Beyond target,False,9,1.305576,5,8,5.181850,5.000918,1.305576
5,2024-06,Beyond target,True,21,1.303051,5,21,10.303284,9.962794,1.303051
2,2024-03,Recovered,True,12,1.294794,5,<NA>,18.188948,17.700029,1.294794
12,2025-09,Recovered,True,12,1.249359,4,<NA>,0.215042,0.216872,1.249359



── Portfolio Summary ───────────────────────────────────────


,Metric,Value,Unit
0,Portfolio Recovery Rate,1.26,x
1,Recovered Cohort Share,95.28,% of collections
2,Top-3 Cohort Share,26.83,% of collections
3,Seasoned Cohorts (12M+),8.00,cohorts
4,Cohorts Above 1.0x,12.00,cohorts
5,Cohorts Above 1.3x,8.00,cohorts


  ✓  cleaned_installments.csv                (11,252 rows)
  ✓  excluded_rows.csv                       (4 rows)
  ✓  moic_table.csv                          (185 rows)
  ✓  cohort_summary.csv                      (15 rows)
  ✓  executive_insights.csv                  (15 rows)
  ✓  portfolio_summary.csv                   (6 rows)
  ✓  cumulative_moic_matrix.csv              (15 rows)
  ✓  payment_moic_matrix.csv                 (15 rows)


## Section 4 · MOIC Curve Charts

`plot_moic_curves`, `plot_net_moic_curves` and `plot_executive_dashboard` share
one curve renderer and theme, and write PNGs into the output folder.

In [5]:
plot_moic_curves(moic_df, f"{out_path}/moic_curve.png")
plot_net_moic_curves(moic_df, f"{out_path}/net_moic_curve.png")
plot_executive_dashboard(moic_df, summary, f"{out_path}/moic_dashboard.png")

  Saved → /tmp/nbout/moic_curve.png


  Saved → /tmp/nbout/net_moic_curve.png


  Saved → /tmp/nbout/moic_dashboard.png
